In [3]:
from poolparty import KMutationPool, KmerPool

# Define the DNA sequence
promoter_wt = "AATTAATGTGAGTTAGCTCACTCATTAGGCACCCCAGGCTTTACACTTTATGCTTCCGGCTCGTATGTTGTGTGG"

# Create pool for single-mutation variants
promoter_pool = KMutationPool(promoter_wt, alphabet='dna', k=1, mode='sequential')

# Create pool for random 10 nt barcode
barcode_pool = KmerPool(length=10, alphabet='dna', mode='random')

# Concatenate mutation with barcode
library_pool = promoter_pool + '...' + barcode_pool + '...'

# Generate all single mutants, each with a random barcode
seqs = library_pool.generate_library(num_complete_iterations=5)

# Print total number of sequences
print(f"Total sequences generated: {len(seqs)}")

# Print first 5 sequences
print("\nFirst 5 sequences:")
for seq in seqs[:5]:
    print(seq)

# Print last 5 sequences  
print("\nLast 5 sequences:")
for seq in seqs[-5:]:
    print(seq)


Total sequences generated: 1125

First 5 sequences:
CATTAATGTGAGTTAGCTCACTCATTAGGCACCCCAGGCTTTACACTTTATGCTTCCGGCTCGTATGTTGTGTGG...TACCATTGTC...
GATTAATGTGAGTTAGCTCACTCATTAGGCACCCCAGGCTTTACACTTTATGCTTCCGGCTCGTATGTTGTGTGG...CACATAGTCG...
TATTAATGTGAGTTAGCTCACTCATTAGGCACCCCAGGCTTTACACTTTATGCTTCCGGCTCGTATGTTGTGTGG...ACTATTCACA...
ACTTAATGTGAGTTAGCTCACTCATTAGGCACCCCAGGCTTTACACTTTATGCTTCCGGCTCGTATGTTGTGTGG...CTGCTCCGCT...
AGTTAATGTGAGTTAGCTCACTCATTAGGCACCCCAGGCTTTACACTTTATGCTTCCGGCTCGTATGTTGTGTGG...CTGATCGTCA...

Last 5 sequences:
AATTAATGTGAGTTAGCTCACTCATTAGGCACCCCAGGCTTTACACTTTATGCTTCCGGCTCGTATGTTGTGTCG...ATTTACTTTC...
AATTAATGTGAGTTAGCTCACTCATTAGGCACCCCAGGCTTTACACTTTATGCTTCCGGCTCGTATGTTGTGTTG...CTTAGACGAA...
AATTAATGTGAGTTAGCTCACTCATTAGGCACCCCAGGCTTTACACTTTATGCTTCCGGCTCGTATGTTGTGTGA...TGTCGGCATG...
AATTAATGTGAGTTAGCTCACTCATTAGGCACCCCAGGCTTTACACTTTATGCTTCCGGCTCGTATGTTGTGTGC...GATTCTAAAC...
AATTAATGTGAGTTAGCTCACTCATTAGGCACCCCAGGCTTTACACTTTATGCTTCCGGCTCGTATGTTGTGTGT...ACACCGCACT...


In [2]:
from poolparty import MixedPool, ListPool, Pool

# Create some pools
pool1 = ListPool(['AAA', 'TTT'])
pool2 = ListPool(['GGG', 'CCC'])

# Create MixedPool with equal weights (default)
mixed_equal = MixedPool([pool1, pool2])
print('MixedPool with equal weights:')
print(f'  num_states: {mixed_equal.num_states}')
print(f'  is_sequential_compatible: {mixed_equal.is_sequential_compatible()}')

# Sequential iteration
print('  Sequential iteration:')
for i in range(mixed_equal.num_states):
    mixed_equal.set_state(i)
    print(f'    state {i}: {mixed_equal.seq}')

# Create MixedPool with custom weights
mixed_weighted = MixedPool([pool1, pool2], weights=[3.0, 1.0])
print('\nMixedPool with custom weights [3.0, 1.0]:')
print(f'  {repr(mixed_weighted)}')

print('\nSuccess! MixedPool is working correctly.')

MixedPool with equal weights:
  num_states: 4
  is_sequential_compatible: True
  Sequential iteration:
    state 0: AAA
    state 1: TTT
    state 2: GGG
    state 3: CCC

MixedPool with custom weights [3.0, 1.0]:
  MixedPool(2 pools, weights=[3.0, 1.0])

Success! MixedPool is working correctly.


In [3]:
a = ListPool(['AAA', 'TTT', 'GGG', 'CCC'])
b = KMutationPool(a, alphabet='ACGT', k=1)

# Set both pools to sequential mode for combinatorial iteration
a.set_mode('sequential')
b.set_mode('sequential')
sequential_seqs = b.generate_library(num_complete_iterations=1)

# Reset to random mode
a.set_mode('random')
b.set_mode('random')
random_seqs = b.generate_library(num_seqs=10)

print("Sequential sequences:")
print(sequential_seqs)

print("\nRandom sequences:")
print(random_seqs)

Sequential sequences:
['CAA', 'GAA', 'TAA', 'ACA', 'AGA', 'ATA', 'AAC', 'AAG', 'AAT', 'ATT', 'CTT', 'GTT', 'TAT', 'TCT', 'TGT', 'TTA', 'TTC', 'TTG', 'AGG', 'CGG', 'TGG', 'GAG', 'GCG', 'GTG', 'GGA', 'GGC', 'GGT', 'ACC', 'GCC', 'TCC', 'CAC', 'CGC', 'CTC', 'CCA', 'CCG', 'CCT']

Random sequences:
['TGT', 'AAT', 'GGA', 'TGG', 'CTT', 'CTT', 'CAC', 'CTT', 'AAT', 'CCG']


In [ ]:
# Split a 1kb genomic region up into 100 nt segments with 20 nt overlap.
# Then generate mutations at 10% per nucleotide.
# Then add shared primer sequences to each end, with the right primer containing a 20 nt barcode. 

import random
from poolparty import SubseqPool, RandomMutationPool, KmerPool
import textwrap
DNA_ALPHABET = list('ACGT')

genomic_seq = ''.join([random.choice(DNA_ALPHABET) for _ in range(1000)])
segments_pool = SubseqPool(genomic_seq, width=100, shift=80)
variants_pool = RandomMutationPool(segments_pool, alphabet=DNA_ALPHABET, mutation_rate=0.1)
barcodes_pool = KmerPool(length=20, alphabet=DNA_ALPHABET)
left_fixed_seq = 'GGTCTCGTATGCCGTCTTCTGCTTG.'
spacer_fixed_seq = '.GGTCTCGTATGCCGTCTTCTGCTTG.'
right_fixed_seq = '.GGTCTCGTATGCCGTCTTCTGCTTG'

oligo_pool = left_fixed_seq + variants_pool + spacer_fixed_seq + barcodes_pool + right_fixed_seq

print("Genomic sequence:")
for line in textwrap.wrap(genomic_seq, width=80):
    print(line)

print("\nSequences in segments_pool:")
seqs = segments_pool.generate_library(num_seqs=10)
for i, seq in enumerate(seqs, 1):
    print(f"Segment {i}: {seq}")

print("\nSequences in variants_pool:")
seqs = variants_pool.generate_library(num_seqs=10)
for i, seq in enumerate(seqs, 1):
    print(f"Variant {i}: {seq}")
    
print("\nSequences in barcodes_pool:")
seqs = barcodes_pool.generate_library(num_seqs=10)
for i, seq in enumerate(seqs, 1):
    print(f"Barcode {i}: {seq}")

    
seqs = oligo_pool.generate_library(num_seqs=10)
print("\nSequences in oligo_pool:")
for i, seq in enumerate(seqs, 1):
    print(f"Oligo {i}: {seq}")


results = oligo_pool.generate_library(num_complete_iterations=1, return_computation_graph=True)
print("\nComputation graph:")
visualize_computation_graph(results['graph'], results['node_sequences'], show_first_only=True)

Genomic sequence:
CAATGTCGGCCTAGCAGAGGTACAGGGGCGAAGGCACGTTGATGTTGATGAATGTTTGTGGCCGTGTACTATGTCGGTCA
CAGTAGAAGTGCTAACTATCCAACATTAAGATGATCATCAGTATGAGCGGCGGGCTGTTTGTGATCAATAACCTGAATAG
CTAACACGTACACCGAACGCTGGTCGGTCACCGTATCGGCACGCCTCCGCACCACGCTAATCTCTTGGGTTTTATCATAA
GTCACACTACCTCTGGTACTTATGCAAATCCGTGATGACTGACCGCCGGAGTTGGAGACCTTCAACGCGGACATTATGGA
AACAGCATCCTTGGAACGAGTCCGGCTCGCGTCTACGATATGCATCGACTAGGCTAATGCCGGCGATCTGTTGGGGTGTG
CCCTTCATTTTAGCAACCCGTGCGCGTCGCAGACTCGACGATCTTGTTACGCACGTCAATTTTGTCTTGAAAGCCCACAA
CTTCCGAGGGAGTAAACGGCGTAGAGTGGACGGCAGTATTGGTTCGCTTGGTTACGATTAACTCGTAAGTCTCTGCCACG
CAACCACGCGCCTGCCACGTGCAGACTGGTACTCCGAGGTAAGAGGGTCTTTTCAAAGCGACGAATACCGTAGCCGATGG
AGTCTTGAAGCGAGGCCGAGATGAGTAACTCCGAATATTGCCCCATCGTCACCTCTGTTGATACTTCGTGGCTGAGTGAG
TATGATAAGACTTATCGTATTGCCAAAGTAAGTCAATTCGTGGGTGTATTACATGCCGTTCCTCTCTTTATCCGTTACTT
AGAATCTAAAGACAATTGAGCGTCCTTCCACACTCGATTTTAATGGCGTGCACGATGCTTAGGCGATTCCCTCTGCATAA
CGCCGTTACTCGGTGATCTAGTCTGGAGCGACTCTCGCTGGCAAACACTATTGATTCCCTTAGCACGAGCATTCGACCCA
TTCATCTCAT